LangChain中提供了一个新的功能模块langchain-mcp-adapters来支持MCP服务

In [ ]:
pip install langchain-mcp-adapters

接下来接入MCP服务也相当简单，只要增加MCP配置文件即可

SSE接入MCP服务
"mcpServers": {
    "amap-amap-sse": {
      "url": "https://mcp.amap.com/sse?key=您在高德官网上申请的key",
      "transport":"streamable_http"
    }
  }
STDIO接入MCP服务(确保已安装 Node.js)
"mcpServers": {
    "amap-maps": {
      "command": "npx",
      "args": ["-y","@amap/amap-maps-mcp-server"],
      "env": {
        "AMAP_MAPS_API_KEY": "您在高德官网上申请的key"
      },
      "transport": "stdio"
    }
  }

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain.community.chat_models import ChatTongyi
# 构建阿里云百炼大模型客户端
llm = ChatTongyi(
    model="qwen-plus",
    api_key=load_key("BAILIAN_API_KEY"),
)
# 相比Cline客户端配置，只要增加transport属性即可。不过测试streamable_http有问题，不知道是不是版本的原因。
client = MultiServerMCPClient(
    {
        # sse接入方式
        #"amap-amap-sse": {
        #    "url": "https://mcp.amap.com/sse?key=451ad40d0e39453600f2a305e31eabe4",
        #    "transport": "streamable_http"
        #},
        # stdio接入方式
        "amap-maps": {
            "command": "npx",
            "args": [
                "-y",
                "@amap/amap-maps-mcp-server"
            ],
            "env": {
                "AMAP_MAPS_API_KEY": "451ad40d0e39453600f2a305e31eabe4"
            },
            "transport": "stdio"
        }
    }
)

tools = await client.get_tools()
agent = create_react_agent(
    model=llm,
    tools=tools
)
response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "帮我规划一条从长沙梅溪湖到溁湾镇的自驾路线"}]},
)
response